# Prompt 09: Prompt Optimization

1. Manual variant sweep
2. LLM-as-proposer (APE-style) candidate generation
3. Held-out overfitting check
4. Sketch of a DSPy pipeline
5. Exercises: optimize one of your real prompts

Deps: `pip install anthropic openai`  (optional: `pip install dspy-ai`)

In [ ]:
import os, re, random

def call(system, user, temperature=0):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=400, temperature=temperature,
            system=system, messages=[{'role':'user','content':user}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=400, temperature=temperature,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content
    return '[no provider]'

## 1. Eval Set: Sentiment Classification

In [ ]:
data = [
    ('Absolutely love it, best purchase this year.',           'positive'),
    ('Broke after two uses. Terrible.',                         'negative'),
    ('It works as advertised, nothing more.',                   'neutral'),
    ('Packaging was fine but setup was a nightmare.',           'negative'),
    ('Arrived on time. Quality seems average for the price.',   'neutral'),
    ('Wonderful service, will buy again.',                      'positive'),
    ('Returned it immediately — totally defective.',           'negative'),
    ('Decent. Not amazing, not awful.',                         'neutral'),
    ('I\'m a fan. Great value.',                                'positive'),
    ('Waste of money. Do not recommend.',                       'negative'),
    ('The product is okay for the price I paid.',               'neutral'),
    ('Excellent packaging and fast shipping.',                  'positive'),
]
random.seed(0)
random.shuffle(data)
dev, holdout = data[:8], data[8:]   # small split for demo

def score(prompt_template):
    hits = 0
    for text, gold in dev:
        out = call('You classify sentiment. Respond with only one word: positive, negative, or neutral.',
                   prompt_template.format(text=text)).strip().lower()
        hits += int(out.startswith(gold))
    return hits / len(dev)

## 2. Manual Variant Sweep

In [ ]:
variants = {
    'v0_vague':       'What is the sentiment? {text}',
    'v1_simple':      'Classify sentiment (positive/negative/neutral): {text}',
    'v2_specific':    'Classify the review sentiment as exactly one of: positive, negative, or neutral.\n\nReview: {text}\nLabel:',
    'v3_fewshot':     '''Classify sentiment as positive, negative, or neutral.

Review: "Great product!"  -> positive
Review: "Totally broken." -> negative
Review: "It works fine." -> neutral

Review: "{text}" ->''',
}

for name, tmpl in variants.items():
    print(f'{name:<15} dev accuracy = {score(tmpl):.2f}')

## 3. LLM-as-Proposer (APE-Style)

In [ ]:
proposer_sys = ('You propose candidate prompts for a classification task. Given examples, '
                'write concise prompt templates that use {text} as the placeholder and are likely '
                'to produce exactly one-word outputs: positive, negative, or neutral.')

examples_str = '\n'.join(f'- "{t}" -> {lbl}' for t, lbl in dev[:4])
proposer_user = f'Examples:\n{examples_str}\n\nReturn 5 distinct prompt templates, numbered.'

raw = call(proposer_sys, proposer_user, temperature=0.9)
proposed = [re.sub(r'^\s*\d+\.\s*','', line).strip().strip('"')
            for line in raw.split('\n')
            if re.match(r'^\s*\d+\.', line) and '{text}' in line]
print(f'proposer produced {len(proposed)} candidates:\n')
for p in proposed: print(' -', p)

scored = [(p, score(p)) for p in proposed]
scored.sort(key=lambda x: -x[1])

print('\nranked (dev):')
for p, s in scored: print(f'  {s:.2f}  {p}')

## 4. Overfitting Check on Held-Out

In [ ]:
def holdout_score(prompt_template):
    hits = 0
    for text, gold in holdout:
        out = call('You classify sentiment. Respond with only one word: positive, negative, or neutral.',
                   prompt_template.format(text=text)).strip().lower()
        hits += int(out.startswith(gold))
    return hits / len(holdout)

print('dev -> holdout generalization:')
for name, tmpl in variants.items():
    print(f'{name:<15}  dev={score(tmpl):.2f}  holdout={holdout_score(tmpl):.2f}')

if scored:
    best_proposed, _ = scored[0]
    print(f'\nbest proposed  dev={score(best_proposed):.2f}  holdout={holdout_score(best_proposed):.2f}')

## 5. DSPy Sketch (Illustrative)

Run if `dspy-ai` is installed; otherwise read the structure.

In [ ]:
try:
    import dspy

    class Classify(dspy.Signature):
        """Classify sentiment as positive, negative, or neutral."""
        review: str = dspy.InputField()
        sentiment: str = dspy.OutputField(desc='exactly one of: positive, negative, neutral')

    # Configure once — use any OpenAI-compatible endpoint or provider.
    # dspy.configure(lm=dspy.LM("openai/gpt-4o-mini", max_tokens=30))

    classify = dspy.Predict(Classify)

    trainset = [dspy.Example(review=t, sentiment=lbl).with_inputs('review') for t, lbl in dev]

    def metric(gold, pred, trace=None):
        return str(gold.sentiment).lower() == str(pred.sentiment).lower().strip()

    # compiled = dspy.BootstrapFewShot(metric=metric).compile(classify, trainset=trainset)
    print('dspy installed — uncomment configure + compile to run a real optimization.')
except ImportError:
    print('dspy-ai not installed. `pip install dspy-ai` to run. Structure to study:')
    print('''
class Classify(dspy.Signature):   # typed I/O
    review: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive|negative|neutral")

classify = dspy.Predict(Classify)
compiled = dspy.BootstrapFewShot(metric=...).compile(classify, trainset=examples)
# compiled now has auto-chosen few-shot demonstrations and optimized instructions.
''')

## 6. Exercise: Optimize One of Your Prompts

Pick a real prompt with measurable output. Build:

1. **Labeled data** — 50+ examples split 70/30 dev/holdout.
2. **A score function.** Exact match, F1, or LLM-judge score.
3. **5 manual variants** — vary wording, format, shot count.
4. **10 LLM-proposed variants** — use a proposer prompt.
5. **Ranking table** — dev + holdout scores.
6. **Ablation** — take the winner; remove each clause; see what matters.

Pick the shortest winning variant that generalizes to holdout.

## 7. Takeaways

- **Prompt optimization = search over prompt space** against a metric.
- **Manual sweep is the first move.** 5 variants, one change at a time.
- **APE-style LLM proposers** widen the search cheaply.
- **DSPy** is the right tool for multi-stage pipelines.
- **Always hold out** data; prompts can and do overfit.
- **Re-optimize on model upgrades.**

Next: **10 — Testing & Evaluation of Prompts** — CI-grade discipline.